# Gemma 4 31B — Language-U V12.5 Atlas-Guided Cortical Cascade (Step 200 -> 400 Continuation + 720-Step Assistant)

- **Resumes Directly From Verified V12 Step-200 Checkpoint (`/content/drive/MyDrive/gemma4_v12_atlas_cascade/gemma4_31b_v12_atlas_cascade.t16c`):** Unpacks bit-exact bitplanes, scales, biases, `inv_s`, Int8 low-rank adapters, RMSNorms, Vision encoder, and 3-bit `embed_tokens` in ~15s (skipping the 40-minute Stage-2 HQQ/SpinQuant pass).
- **Continues Dual-Regime E2E Logit-KL (`Steps 201 -> 400`) + Fixed `cca_tier2_bypass` Every Step:** Trains both 60/60 Dense and `16/60` Core-Protected + `L16,18,40,45` Bypassed on every optimizer step (`8x2`) with smooth AdamW re-warmup and a 720-step Target-Shared EAGLE-3 Assistant (`Q4`/`Q2`) + `BHDR-256`.
- **Outputs saved & SHA-256 verified in `/content/drive/MyDrive/gemma4_v12_5_atlas_cascade` + automatic Zero-CU `drive.flush_and_unmount()` + `runtime.unassign()`.**

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

# 1. Instant Hardware Envelope Guard
assert torch.cuda.is_available(), (
    "No CUDA GPU detected! Go to Runtime -> Change runtime type -> select A100 GPU (or L4 GPU) + High-RAM, then click Run all."
)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
gpu_name = torch.cuda.get_device_name(0)
meminfo_kb = int([l.split()[1] for l in Path("/proc/meminfo").read_text().splitlines() if l.startswith("MemTotal:")][0])
sys_ram_gb = meminfo_kb / (1024**2)
print(f"[Hardware Check] GPU={gpu_name} ({vram_gb:.2f} GiB VRAM) | System RAM={sys_ram_gb:.1f} GiB", flush=True)
assert vram_gb >= 14.0, f"GPU VRAM {vram_gb:.1f} GiB < 14 GiB minimum."
assert sys_ram_gb >= 25.0, f"System RAM {sys_ram_gb:.1f} GiB < 25 GiB (turn ON High-RAM)."

# 2. Mount Drive, Verify Step-200 V12 Checkpoint Exists, & Sync Repo
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

INIT_T16C = Path("/content/drive/MyDrive/gemma4_v12_atlas_cascade/gemma4_31b_v12_atlas_cascade.t16c")
assert INIT_T16C.is_file() and INIT_T16C.stat().st_size == 6723915520, (
    f"Missing or incomplete Step-200 V12 checkpoint on Drive: {INIT_T16C}"
)
print(f"[Checkpoint Found] {INIT_T16C} ({INIT_T16C.stat().st_size:,} B)", flush=True)

DRIVE_V12_5 = Path("/content/drive/MyDrive/gemma4_v12_5_atlas_cascade")
DRIVE_V12_5.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
print("[Repo Commit]", subprocess.check_output(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True).strip(), flush=True)

# 3. Dependencies & 6-Gate Self-Test
need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    need_pip = True
if need_pip:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=5.0.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v12_5_atlas_cascade_gpu.py"
res = subprocess.run([sys.executable, str(SCRIPT), "--selftest"], text=True, capture_output=True)
print(res.stdout, res.stderr, flush=True)
res.check_returncode()

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v12_5_atlas_cascade_gpu.py"
LOG_PATH = Path("/content/v12_5_train.log")
INIT_T16C = Path("/content/drive/MyDrive/gemma4_v12_atlas_cascade/gemma4_31b_v12_atlas_cascade.t16c")
DRIVE_V12_5 = Path("/content/drive/MyDrive/gemma4_v12_5_atlas_cascade")

env = os.environ.copy()
env["V12_5_DRIVE_DIR"] = str(DRIVE_V12_5)
env["V12_5_OUT_DIR"] = "/content/v12_5_outputs"
env["V12_5_INIT_T16C"] = str(INIT_T16C)
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
env["COLAB_AUTO_UNASSIGN"] = "0"
env["PYTHONUNBUFFERED"] = "1"

with open(LOG_PATH, "w", encoding="utf-8") as log_f:
    proc = subprocess.Popen(
        [
            sys.executable, "-u", str(SCRIPT),
            "--run", "--init-t16c", str(INIT_T16C),
            "--start-step", "200", "--extra-steps", "200",
            "--ast-steps", "720", "--micro-bs", "8", "--grad-accum", "2",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        log_f.write(line)
    rc = proc.wait()

shutil.copy2(LOG_PATH, DRIVE_V12_5 / "v12_5_train.log")
subprocess.run(["sync"], check=False)
assert rc == 0, f"V12.5 pipeline exited with {rc} (log saved to {DRIVE_V12_5 / 'v12_5_train.log'})"
print(subprocess.check_output(["sha256sum", "-c", "SHA256SUMS_V12_5.txt"], cwd=str(DRIVE_V12_5), text=True), flush=True)
print((DRIVE_V12_5 / "v12_5_report.json").read_text(encoding="utf-8"), flush=True)
print("[Zero-CU Guard] All V12.5 SHA-256 checksums verified on Drive; flushing Drive & auto-disconnecting runtime...", flush=True)
from google.colab import drive, runtime
try:
    drive.flush_and_unmount()
except Exception as e:
    print(f"[Drive Flush Note] {e}", flush=True)
runtime.unassign()